In [ ]:
# os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
# os.environ["TF_XLA_FLAGS=--tf_xla_enable_xla_devices"] = "false"
import glob

import tensorflow as tf

from train.models import FullModel
from util import dataset as u_dataset
from util import image as u_image

In [ ]:
def get_dataset(directory):
    raw_dataset = tf.data.TFRecordDataset(directory)

    feature_description = {
        "image": tf.io.FixedLenFeature([], tf.string),
        "camera": tf.io.FixedLenFeature([], tf.string),
        "intrinsics": tf.io.FixedLenFeature([], tf.string),
        "objectness": tf.io.FixedLenFeature([], tf.string),
        "offsets": tf.io.FixedLenFeature([], tf.string),
        "loss_mask": tf.io.FixedLenFeature([], tf.string),
    }

    def _parse_tensor(serialized_tensor):
        return {
            "image": tf.ensure_shape(
                tf.io.parse_tensor(serialized_tensor["image"], out_type=tf.uint8), [480, 320, 4]
            ),
            "camera": tf.ensure_shape(
                tf.io.parse_tensor(serialized_tensor["camera"], out_type=tf.float32), [3]
            ),
            "intrinsics": tf.ensure_shape(
                tf.io.parse_tensor(serialized_tensor["intrinsics"], out_type=tf.float32), [4]
            ),
            "objectness_mask": tf.ensure_shape(
                tf.io.parse_tensor(serialized_tensor["objectness"], out_type=tf.float32), [15, 20]
            ),
            "offsets": tf.ensure_shape(
                tf.io.parse_tensor(serialized_tensor["offsets"], out_type=tf.float32), [15, 20, 2]
            ),
            "loss_mask": tf.ensure_shape(
                tf.io.parse_tensor(serialized_tensor["loss_mask"], out_type=tf.float32), [15, 20]
            ),
        }

    def _parse_function(example_proto):
        # Parse the input tf.train.Example proto using the dictionary above.
        return _parse_tensor(tf.io.parse_single_example(example_proto, feature_description))

    return raw_dataset.map(_parse_function)

In [ ]:
def get_data_info(directory="data"):
    """Get all .tfrecords files and the number of all samples across all selected data files.

    Args:
        directory: The directory of the .tfrecords files used for training.

    Returns:
        A dict where "file_names" is a list of paths to .tfrecords files and "num_samples is the number of samples across all data files
    """
    file_names = []
    num_samples = 0

    for file in glob.glob(f"./{directory}/*.tfrecords"):
        file_names.append(file)

        # Count samples
        num_samples += len(glob.glob(f"{file.removesuffix('.tfrecords')}/*.jpg"))

    return {"file_names": file_names, "num_samples": num_samples}

In [ ]:
test_directory = "data/Joerg_Joerg_CompetitionWalk_GO2025__HULKs_2ndHalf_5"
label = u_dataset.load_labels(test_directory)[165]

offsets, objectness, loss_mask = u_dataset.get_masks(label, "ball")

res_out = tf.shape(offsets)[-3:-1]

scale = tf.cast(((480, 640) / res_out)[::-1], offsets.dtype)
pixels = tf.cast(
    tf.stack(tf.meshgrid(tf.range(res_out[1]), tf.range(res_out[0])), axis=-1),
    offsets.dtype,
)
coords = tf.reshape((offsets + pixels - 0.5) * scale, (1, 20 * 15, 2))

# print(coords)
# print(offsets)

u_image.show_cell_on_image(test_directory, label, "ball", grid_dims=(15, 20))

In [ ]:
data = get_data_info()
dataset = get_dataset(data["file_names"])

epochs = 200
validation_split = 0.3
batch_size = 32
num_samples = num_samples = data["num_samples"]
train_samples = round(num_samples * (1 - validation_split))
val_samples = round(num_samples * validation_split)

print("Number of samples: ", num_samples)
print("Train Size: ", train_samples)
print("Val Samples: ", val_samples)

dataset = dataset.shuffle(batch_size, seed=42)

train_ds = dataset.take(train_samples)
val_ds = dataset.skip(val_samples)

train_ds = train_ds.batch(batch_size)
val_ds = val_ds.batch(batch_size)

# To counteract "Local rendezvous warning"
train_ds = train_ds.repeat(-1)
val_ds = val_ds.repeat(-1)

In [ ]:
# Upper camera dimensions. Width is halved because of YUYV format
model = FullModel(480, 320)
model.compile(optimizer=tf.keras.optimizers.Adam())

model.get_layer("encoder").summary()

In [ ]:
model.fit(
    x=train_ds,
    validation_data=val_ds,
    epochs=epochs,
    steps_per_epoch=train_samples // batch_size,
    validation_steps=val_samples // batch_size,
)

In [ ]:
# Save Encoder

model.get_layer("encoder").save("./models/encoder/encoder_overfit.keras")

# # Load Encoder
# patch_e = tf.keras.models.load_model("patch_extractor.keras")

# model.summary()


In [ ]:
for element in train_ds:
    print(element.keys())
    print(element["image"].shape)
    print(element["camera"].shape)
    print(element["intrinsics"].shape)
    print(element["objectness_mask"].shape)
    print(element["offsets"].shape)
    break

In [ ]:
test_directory = "data/Joerg_Joerg_CompetitionWalk_GO2025__HULKs_2ndHalf_5"

label = u_dataset.load_labels(test_directory)[160]

image = u_dataset.load_image(test_directory, label, image_format=u_image.ImageFormat.YUYV)
camera = u_dataset.camera_from_label(label)
intrinsics = u_dataset.intrinsics_from_label(label)

image_yuv = tf.reshape(tf.constant(image), (-1, 480, 320, 4))
camera = tf.constant(camera, dtype=tf.float32)
intrinsics = tf.constant(intrinsics, dtype=tf.float32)

print("Image: ", image_yuv.shape)
print("Camera: ", camera)
print("Intrinsics: ", intrinsics)

results = model((image_yuv, camera, intrinsics), training=None)

In [ ]:
# print(results["ball"][1])

# camera_rays, camera_rotation, rotated_camera_rays, camera_height, factors, distances_in_camera, positions_in_camera, pixel_sizes

camera_rays = results["ball"][3]
camera_rotation = results["ball"][4]
rotated_camera_rays = results["ball"][5]
camera_height = results["ball"][6]
factors = results["ball"][7]
distances_in_camera = results["ball"][8]
positions_in_camera = results["ball"][9]
pixel_sizes = results["ball"][10]
coords_results = results["ball"][11]
intrinsics_results = results["ball"][12]
masks = results["ball"][1]

# print("Intrinsics: ", intrinsics_results)
print("Factors: ", factors)
# print("Distances: ", distances_in_camera)
print("Camera rays: ", camera_rays)
print("Rotated Camera Rays: ", rotated_camera_rays)
print("Positions: ", positions_in_camera)
# print("Camera Height: ", camera_height)
print("Camera Rotation: ", camera_rotation)
# print("Coords: ", coords_results)
print("Masks: ", masks)


u_image.show_patches_on_image(image, "ball", results)

In [ ]:
print(model.encoder.output_shape)
print(model.encoder.input_shape)

print(model.categories["ball"])